# Time-Binned Geomagnetic Activity Dataset

**One row = one fixed time bin (3 hours).** Instead of anchoring on CME events, we put every dataset on a single regular clock and frame the problem as time-series forecasting:

> *Given the recent history of solar-wind conditions, what will the geomagnetic activity (ap) be N hours from now?*

**Datasets used**
- `omni_solar_wind_parameters` — hourly solar wind at L1; the source of both features (Bz, speed, density, pressure, IMF) **and** the target (`ap_index_nt`).
- `solar_flare_events` — collapsed to per-bin counts + max GOES class.
- `space_weather_events` (CME rows) — collapsed to per-bin counts + max speed.

**Leakage guardrails baked in (see each step):**
1. Rolling features are **trailing only** (`closed='right'`, never `center=True`).
2. The target is built with a forward `shift(-HORIZON)`; features never reference a bin ≥ the target's time.
3. Train/test split is **temporal** (earlier → train, later → test); no shuffling.
4. Gaps are **forward-filled only** (never back-filled or interpolated forward).
5. A **persistence baseline** is included — the real bar to beat, not the mean.
6. Scaling/imputation must be fit on **train only** (done in the modeling pipeline).

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

ds_omni   = load_dataset("juliensimon/omni-solar-wind-parameters", split="all")
ds_flares = load_dataset("juliensimon/solar-flare-events", split="all")
ds_events = load_dataset("juliensimon/donki-space-weather-events", split="all")

df_omni   = ds_omni.to_pandas()
df_flares = ds_flares.to_pandas()
df_events = ds_events.to_pandas()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

- `BIN` = 3h grid (matches how Kp/ap are reported).
- `WINDOW_BINS` = how many trailing bins go into each feature window (8 bins = 24h of history).
- `HORIZONS` = the forecast horizons, in hours → bins. We build a target for each so one dataset supports 3h / 6h / 12h / 24h forecasts. L1 solar wind gives real physical skill at short horizons (~1h transit + geomagnetic response); 24h-ahead skill mostly comes from recurrence, so expect the short horizons to look best.
- `STORM_THRESHOLD` = ap level that defines a "storm" bin for the classification targets (ap ≥ 50 ≈ Kp 5, minor storm).
- `START` = 1995 (WIND era; OMNI wind + ap are good from here). The event catalogs start later — flares 2017-02, CMEs (DONKI) 2010 — so before each catalog begins its columns are **NaN** ("not observed"), never 0 ("observed nothing happened"). Tree models treat NaN as its own category, so they can distinguish the two.

In [2]:
BIN          = '3h'
WINDOW_BINS  = 8     # 8 * 3h = 24h trailing history per feature
CME_WINDOW_BINS = 24 # 24 * 3h = 72h trailing window for CME features (sun-to-Earth travel time is ~1-3 days)
HORIZONS     = {3: 1, 6: 2, 12: 4, 24: 8}   # hours ahead -> bins ahead
STORM_THRESHOLD = 50  # ap >= 50 ~ Kp 5 (minor storm) for the classification targets
GAP_FFILL_LIMIT = 4  # forward-fill at most 4 bins (12h) across a data gap

# 1995 = start of the WIND-spacecraft era: OMNI wind/ap quality is good from here,
# and wind + ap history is where all the model skill comes from. The flare (2017-)
# and CME (2010-) catalogs start later; before those dates their columns are NaN
# ("not observed"), never 0 ("observed nothing") — see steps 2-3.
START = '1995-01-01'
END   = '2024-12-31'

WIND_COLS = ['bz_gsm_nt', 'b_magnitude_avg_nt', 'flow_speed_kms',
             'proton_density_cm3', 'flow_pressure_npa', 'electric_field_mvpm']
TARGET_COL = 'ap_index_nt'

## 1. Build the regular time grid + bin the solar wind

OMNI is hourly. We resample it onto the 3h grid by averaging the wind within each bin, and take the **max** ap in each bin (storm-relevant). `resample` produces a continuous, gap-aware grid — missing bins become NaN rather than silently vanishing.

In [3]:
df_omni['datetime'] = pd.to_datetime(df_omni['datetime'])
df_omni = (df_omni[(df_omni['datetime'] >= START) & (df_omni['datetime'] <= END)]
           .set_index('datetime')
           .sort_index())

# Within-bin aggregation: mean for wind features, max for the ap target.
grid = df_omni.resample(BIN).agg({**{c: 'mean' for c in WIND_COLS},
                                  TARGET_COL: 'max'})

print(f"Grid: {len(grid):,} bins  |  {grid.index.min()} -> {grid.index.max()}")
print(f"Missing-bin rate per column:\n{grid.isna().mean().round(3)}")
grid.head()

Grid: 87,657 bins  |  1995-01-01 00:00:00 -> 2024-12-31 00:00:00
Missing-bin rate per column:
bz_gsm_nt              0.002
b_magnitude_avg_nt     0.002
flow_speed_kms         0.004
proton_density_cm3     0.019
flow_pressure_npa      0.019
electric_field_mvpm    0.004
ap_index_nt            0.000
dtype: float64


,bz_gsm_nt,b_magnitude_avg_nt,flow_speed_kms,proton_density_cm3,flow_pressure_npa,electric_field_mvpm,ap_index_nt
datetime,,,,,,,
1995-01-01 00:00:00,-0.633333,3.400000,316.666667,18.133333,3.203333,0.200000,4.0
1995-01-01 03:00:00,0.533333,4.533333,315.000000,15.633333,2.736667,-0.170000,0.0
1995-01-01 06:00:00,0.500000,4.300000,321.333333,17.500000,3.163333,-0.156667,3.0
1995-01-01 09:00:00,-1.866667,4.400000,322.666667,17.433333,3.190000,0.600000,5.0
1995-01-01 12:00:00,1.166667,3.433333,318.000000,17.300000,3.203333,-0.366667,2.0


## 2. Bin the flares — counts + max class per bin

Flares collapse to two per-bin features: how many flares peaked in the bin, and the strongest GOES class. Joined on the bin timestamp (no fuzzy tolerance).

In [4]:
class_map = {'A': 1, 'B': 2, 'C': 3, 'M': 4, 'X': 5}

f = df_flares.copy()
f['peak_time'] = pd.to_datetime(f['peak_time'])
f = f.dropna(subset=['peak_time'])
f = f[(f['peak_time'] >= START) & (f['peak_time'] <= END)]
f['flare_class_num'] = f['goes_class_letter'].map(class_map)
f = f.set_index('peak_time').sort_index()

flare_binned = f.resample(BIN).agg(
    flare_count=('flare_class_num', 'size'),
    flare_max_class=('flare_class_num', 'max'),
)
# size counts rows even when class is NaN; recount only valid-class flares
flare_binned['flare_count'] = f['flare_class_num'].resample(BIN).count()
flare_binned = flare_binned.reindex(grid.index).fillna(0)

# Before the flare catalog begins, "no flare recorded" means "not observed",
# not "nothing happened" — keep those bins NaN so the model can tell them apart.
FLARE_DATA_START = f.index.min()
flare_binned.loc[flare_binned.index < FLARE_DATA_START] = np.nan
print(f"Flare catalog coverage starts {FLARE_DATA_START}  "
      f"({(flare_binned.index < FLARE_DATA_START).mean():.0%} of bins are pre-catalog -> NaN)")

flare_binned.describe()

Flare catalog coverage starts 2017-02-09 00:50:00  (74% of bins are pre-catalog -> NaN)


,flare_count,flare_max_class
count,23056.000000,23056.000000
mean,0.659568,1.034351
std,1.079711,1.433993
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,3.000000
max,7.000000,5.000000


## 3. Bin the CMEs — count, speed, width, Earth-directedness per bin

CME launches carry more usable signal than just speed:
- `cme_half_angle_deg` — angular width. Wide (~halo) CMEs are far more likely to hit Earth and drive storms.
- `cme_latitude` / `cme_longitude` — where on the sun it launched. A CME aimed near disk center (small |lat|, |lon|) is Earth-directed; one from the limb usually misses us.

Per bin we keep: launch count, max speed, max half-angle, and the count of *Earth-directed* launches (|lon| ≤ 45°, |lat| ≤ 30°, using launches with known direction).

These are still *launch* features: the CME's own Bz isn't knowable at launch, so they can't say how geoeffective an impact will be — but they do say something is likely inbound, which matters most at the 12–24h horizons.

In [5]:
c = df_events[df_events['event_type'] == 'CME'].copy()
c['start_time'] = pd.to_datetime(c['start_time'], utc=True).dt.tz_localize(None)
c = c[(c['start_time'] >= START) & (c['start_time'] <= END)]

# Earth-directed = launched near disk center (longitude within 45 deg, latitude within 30 deg).
# Longitude is missing for ~14% of CMEs; those count as not-Earth-directed (unknown direction).
c['earth_directed'] = ((c['cme_longitude'].abs() <= 45) &
                       (c['cme_latitude'].abs() <= 30)).astype(float)

c = c.set_index('start_time').sort_index()

cme_binned = c.resample(BIN).agg(
    cme_count=('cme_speed_kms', 'size'),
    cme_max_speed=('cme_speed_kms', 'max'),
    cme_max_half_angle=('cme_half_angle_deg', 'max'),
    cme_earthdir_count=('earth_directed', 'sum'),
)
cme_binned = cme_binned.reindex(grid.index).fillna(0)

# Same rule as flares: bins before the DONKI catalog begins are "not observed" -> NaN.
CME_DATA_START = c.index.min()
cme_binned.loc[cme_binned.index < CME_DATA_START] = np.nan
print(f"CME catalog coverage starts {CME_DATA_START}  "
      f"({(cme_binned.index < CME_DATA_START).mean():.0%} of bins are pre-catalog -> NaN)")

cme_binned.describe()

CME catalog coverage starts 2010-04-03 09:54:00  (51% of bins are pre-catalog -> NaN)


,cme_count,cme_max_speed,cme_max_half_angle,cme_earthdir_count
count,43085.000000,43085.000000,43085.000000,43085.000000
mean,0.169224,79.139847,4.099350,0.021446
std,0.434483,221.804477,10.870126,0.150216
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,5.000000,3529.000000,92.000000,2.000000


## 4. Assemble the panel + forward-fill gaps (guardrail #4)

Join everything on the bin timestamp. Then **forward-fill only** the wind columns across short gaps — forward-fill uses past values, so it cannot leak the future. We cap the fill length so a long outage doesn't get smeared over (`GAP_FFILL_LIMIT`). We deliberately do **not** fill the target.

In [6]:
panel = grid.join(flare_binned).join(cme_binned)

# Forward-fill ONLY the continuous wind features, only across short gaps.
panel[WIND_COLS] = panel[WIND_COLS].ffill(limit=GAP_FFILL_LIMIT)

print(f"Panel shape: {panel.shape}")
print(f"Remaining NaN rate (wind):\n{panel[WIND_COLS].isna().mean().round(3)}")

Panel shape: (87657, 13)
Remaining NaN rate (wind):
bz_gsm_nt              0.001
b_magnitude_avg_nt     0.001
flow_speed_kms         0.001
proton_density_cm3     0.009
flow_pressure_npa      0.009
electric_field_mvpm    0.001
dtype: float64


## 5. Trailing-window features (guardrail #1)

For each wind column we compute rolling statistics over the last `WINDOW_BINS` bins. `rolling` is **trailing by default** (`closed='right'`, includes the current bin, never future ones). We add:
- `*_mean`, `*_min`, `*_max` over the window
- `*_last` = the current bin's value
- `*_trend` = current value minus the value at the start of the window (rate of change)

The `_now` snapshot of recent measured Bz / E-field is exactly the info that *is* predictive a few hours ahead — the whole reason this framing has skill where the CME-launch one doesn't.

**ap gets the same trailing-history treatment.** ap is strongly autocorrelated, so its own recent mean/max/trend is likely the single most informative block of features — it tells the model whether a storm is building, peaking, or recovering, which raw persistence (`ap_now`) alone can't distinguish.

In [7]:
feat = pd.DataFrame(index=panel.index)

for col in WIND_COLS:
    s = panel[col]
    roll = s.rolling(WINDOW_BINS, min_periods=WINDOW_BINS // 2)  # trailing
    feat[f'{col}_mean']  = roll.mean()
    feat[f'{col}_min']   = roll.min()
    feat[f'{col}_max']   = roll.max()
    feat[f'{col}_last']  = s                          # current bin value
    feat[f'{col}_trend'] = s - s.shift(WINDOW_BINS - 1)  # change across window

# Rectified southward Bz: only the storm-driving (southward, Bz<0) component.
# Averaging raw Bz lets north/south cancel; this can't be washed out.
bz_south = (-panel['bz_gsm_nt']).clip(lower=0)
bs_roll = bz_south.rolling(WINDOW_BINS, min_periods=WINDOW_BINS // 2)
feat['bz_south_mean'] = bs_roll.mean()
feat['bz_south_max']  = bs_roll.max()
feat['bz_south_last'] = bz_south

# Coupling function: speed x southward Bz ~ the dawn-dusk electric field
# that actually drives energy into the magnetosphere.
coupling = panel['flow_speed_kms'] * bz_south
cp_roll = coupling.rolling(WINDOW_BINS, min_periods=WINDOW_BINS // 2)
feat['coupling_mean'] = cp_roll.mean()
feat['coupling_max']  = cp_roll.max()

# Flare features: count over the trailing 24h + current-bin max intensity
feat['flare_count_win']     = panel['flare_count'].rolling(WINDOW_BINS, min_periods=1).sum()
feat['flare_max_class_now'] = panel['flare_max_class']

# CME features over a trailing 72h window (CME_WINDOW_BINS), not 24h:
# a CME takes ~1-3 days to reach Earth, so the launch that matters for the
# 12-24h targets happened well before the standard 24h feature window.
cme_roll = lambda col, how: getattr(
    panel[col].rolling(CME_WINDOW_BINS, min_periods=1), how)()
feat['cme_count_72h']          = cme_roll('cme_count', 'sum')
feat['cme_earthdir_count_72h'] = cme_roll('cme_earthdir_count', 'sum')
feat['cme_max_speed_72h']      = cme_roll('cme_max_speed', 'max')
feat['cme_max_half_angle_72h'] = cme_roll('cme_max_half_angle', 'max')
feat['cme_count_24h']          = panel['cme_count'].rolling(WINDOW_BINS, min_periods=1).sum()
feat['cme_max_speed_now']      = panel['cme_max_speed']

# ap history: persistence value + trailing stats of the target itself.
# All trailing (bins <= T), so no leakage — same guardrail as the wind features.
ap = panel[TARGET_COL]
ap_roll = ap.rolling(WINDOW_BINS, min_periods=WINDOW_BINS // 2)
feat['ap_now']   = ap
feat['ap_mean']  = ap_roll.mean()
feat['ap_max']   = ap_roll.max()
feat['ap_trend'] = ap - ap.shift(WINDOW_BINS - 1)

# 27-day solar-rotation recurrence: the sun rotates every ~27 days, so
# high-speed streams (and the activity they cause) tend to repeat one
# rotation later. shift() looks strictly backward — no leakage.
REC_BINS = int(27 * 24 / 3)   # 27 days = 216 bins
feat['ap_27d_ago']    = ap.shift(REC_BINS)
feat['speed_27d_ago'] = panel['flow_speed_kms'].shift(REC_BINS)

print(f"{feat.shape[1]} features built")
feat.head()

49 features built


,bz_gsm_nt_mean,bz_gsm_nt_min,bz_gsm_nt_max,bz_gsm_nt_last,bz_gsm_nt_trend,b_magnitude_avg_nt_mean,b_magnitude_avg_nt_min,b_magnitude_avg_nt_max,b_magnitude_avg_nt_last,b_magnitude_avg_nt_trend,...,cme_max_speed_72h,cme_max_half_angle_72h,cme_count_24h,cme_max_speed_now,ap_now,ap_mean,ap_max,ap_trend,ap_27d_ago,speed_27d_ago
datetime,,,,,,,,,,,,,,,,,,,,,
1995-01-01 00:00:00,NaN,NaN,NaN,-0.633333,NaN,NaN,NaN,NaN,3.400000,NaN,...,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN
1995-01-01 03:00:00,NaN,NaN,NaN,0.533333,NaN,NaN,NaN,NaN,4.533333,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1995-01-01 06:00:00,NaN,NaN,NaN,0.500000,NaN,NaN,NaN,NaN,4.300000,NaN,...,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN
1995-01-01 09:00:00,-0.366667,-1.866667,0.533333,-1.866667,NaN,4.158333,3.4,4.533333,4.400000,NaN,...,NaN,NaN,NaN,NaN,5.0,3.0,5.0,NaN,NaN,NaN
1995-01-01 12:00:00,-0.060000,-1.866667,1.166667,1.166667,NaN,4.013333,3.4,4.533333,3.433333,NaN,...,NaN,NaN,NaN,NaN,2.0,2.8,5.0,NaN,NaN,NaN


## 6. Forward targets (guardrail #2)

One regression target and one classification target **per horizon**, via `shift(-bins)`:
- `ap_target_{H}h` — max ap in the bin H hours ahead.
- `storm_{H}h` — 1 if that future ap ≥ `STORM_THRESHOLD` (ap 50 ≈ Kp 5). ap is heavy-tailed and MSE-trained regressors ignore the rare storm bins, so the classification framing (evaluated with precision/recall, not accuracy) is what matters operationally.

Every feature at row T uses only bins ≤ T; every target is strictly in the future. We drop rows where any target is missing, so all horizons share the exact same rows and results are directly comparable.

In [8]:
target_cols = []
for hours, bins in HORIZONS.items():
    assert bins > 0, "target must look forward"
    future_ap = panel[TARGET_COL].shift(-bins)
    feat[f'ap_target_{hours}h'] = future_ap
    feat[f'storm_{hours}h'] = (future_ap >= STORM_THRESHOLD).astype('float')
    target_cols += [f'ap_target_{hours}h', f'storm_{hours}h']

# Keep only rows with ap_now and every target present (all horizons comparable).
dataset = feat.dropna(subset=['ap_now'] + [c for c in target_cols if c.startswith('ap_target')]).copy()
for c in target_cols:
    if c.startswith('storm'):
        dataset[c] = dataset[c].astype(int)

n_feat = dataset.shape[1] - len(target_cols)
print(f"Final dataset: {dataset.shape[0]:,} rows x {n_feat} features + {len(target_cols)} targets")
for hours in HORIZONS:
    print(f"  {hours:>2}h ahead: storm rate = {dataset[f'storm_{hours}h'].mean():.2%}  "
          f"(ap_target mean {dataset[f'ap_target_{hours}h'].mean():.1f}, max {dataset[f'ap_target_{hours}h'].max():.0f})")

Final dataset: 87,649 rows x 49 features + 8 targets
   3h ahead: storm rate = 1.83%  (ap_target mean 10.0, max 400)
   6h ahead: storm rate = 1.83%  (ap_target mean 10.0, max 400)
  12h ahead: storm rate = 1.83%  (ap_target mean 10.0, max 400)
  24h ahead: storm rate = 1.83%  (ap_target mean 10.0, max 400)


## 7. Temporal split + baselines per horizon (guardrails #3, #5)

Split by time — first 80% of bins train, last 20% test. Then the baselines that actually matter, per horizon:
- **Regression:** persistence (predict ap stays at `ap_now`) vs the train-mean baseline.
- **Classification:** storm persistence (predict a storm iff it's storming *now*) — reported as precision/recall, since accuracy is meaningless at a ~1-2% storm rate.

Skill should degrade as the horizon grows; a model only counts as useful where it beats the persistence row for that same horizon.

In [9]:
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
                             precision_score, recall_score, f1_score)

dataset = dataset.sort_index()
split = int(len(dataset) * 0.8)
train, test = dataset.iloc[:split], dataset.iloc[split:]
print(f"Train: {len(train):,}  ({train.index.min()} -> {train.index.max()})")
print(f"Test:  {len(test):,}  ({test.index.min()} -> {test.index.max()})")

def evaluate(name, y_true, y_pred):
    print(f"  {name:<22} MAE={mean_absolute_error(y_true, y_pred):6.2f}  "
          f"RMSE={root_mean_squared_error(y_true, y_pred):7.2f}  "
          f"R2={r2_score(y_true, y_pred):.3f}")

for hours in HORIZONS:
    y_test = test[f'ap_target_{hours}h']
    print(f"\n=== {hours}h ahead ===")
    evaluate('Mean baseline', y_test, np.full(len(y_test), train[f'ap_target_{hours}h'].mean()))
    evaluate('Persistence',   y_test, test['ap_now'])  # <-- the real bar

    # Storm classification: persistence = "storming now -> storming then"
    y_cls  = test[f'storm_{hours}h']
    p_cls  = (test['ap_now'] >= STORM_THRESHOLD).astype(int)
    print(f"  {'Storm persistence':<22} "
          f"precision={precision_score(y_cls, p_cls, zero_division=0):.2f}  "
          f"recall={recall_score(y_cls, p_cls, zero_division=0):.2f}  "
          f"F1={f1_score(y_cls, p_cls, zero_division=0):.2f}  "
          f"(base rate {y_cls.mean():.2%})")

Train: 70,119  (1995-01-01 00:00:00 -> 2018-12-30 18:00:00)
Test:  17,530  (2018-12-30 21:00:00 -> 2024-12-30 00:00:00)

=== 3h ahead ===
  Mean baseline          MAE=  7.56  RMSE=  14.04  R2=-0.018
  Persistence            MAE=  4.17  RMSE=   8.96  R2=0.586
  Storm persistence      precision=0.50  recall=0.50  F1=0.50  (base rate 1.16%)

=== 6h ahead ===
  Mean baseline          MAE=  7.56  RMSE=  14.04  R2=-0.018
  Persistence            MAE=  5.43  RMSE=  11.80  R2=0.281
  Storm persistence      precision=0.36  recall=0.36  F1=0.36  (base rate 1.16%)

=== 12h ahead ===
  Mean baseline          MAE=  7.56  RMSE=  14.04  R2=-0.018
  Persistence            MAE=  6.50  RMSE=  14.61  R2=-0.101
  Storm persistence      precision=0.21  recall=0.21  F1=0.21  (base rate 1.16%)

=== 24h ahead ===
  Mean baseline          MAE=  7.56  RMSE=  14.04  R2=-0.018
  Persistence            MAE=  7.48  RMSE=  17.69  R2=-0.615
  Storm persistence      precision=0.06  recall=0.06  F1=0.06  (base rate 1.1

## 8. Save

Saved with the bin timestamp as the index so the temporal ordering is preserved for any downstream modeling. Remember when modeling: **fit the scaler/imputer on `train` only** (guardrail #6) — wrap them in a `Pipeline` and call `.fit` on the training rows alone, and use `TimeSeriesSplit` (not `KFold`) for any tuning.

In [10]:
dataset.to_csv('../data/time_binned_dataset.csv')
print(f"Saved ../data/time_binned_dataset.csv  ({dataset.shape[0]:,} rows, {dataset.shape[1]} cols)")
print(f"\nTargets per horizon: " + ", ".join(f"ap_target_{h}h / storm_{h}h" for h in HORIZONS))
print(f"Feature history: {WINDOW_BINS} bins = {WINDOW_BINS * 3}h trailing")
print("\nWhen modeling: pick ONE horizon's targets and drop the others from X "
      "(they are all future values — using any target column as a feature is leakage).")

Saved ../data/time_binned_dataset.csv  (87,649 rows, 57 cols)

Targets per horizon: ap_target_3h / storm_3h, ap_target_6h / storm_6h, ap_target_12h / storm_12h, ap_target_24h / storm_24h
Feature history: 8 bins = 24h trailing

When modeling: pick ONE horizon's targets and drop the others from X (they are all future values — using any target column as a feature is leakage).
